# Day 4: XGBoost from Scratch

In this notebook, we will build an XGBoost classifier for binary classification. Unlike traditional decision trees, XGBoost improves predictions by sequentially fitting trees to the errors of previous trees. Instead of using Gini impurity, each tree is built using the first and second derivatives of the loss function.

The decision tree implementation has already been provided. Our goal is to understand how these trees are combined into a complete boosting algorithm and how predictions are updated during training.

In [ ]:
import pandas as pd
import numpy as np
from wrapped_models.xg_boost_tree import XGBoostTree

# Log-Odds and Probabilities

For binary classification, XGBoost predicts **log-odds** rather than probabilities directly. Log-odds can take any real value, making optimization much easier than working with probabilities constrained between 0 and 1.

This concept of log-odds is also fundamental to **Logistic Regression**. However, while Logistic Regression calculates $z$ as a simple linear function of features:

$$z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \dots$$

in XGBoost $z$ is computed as the cumulative sum of outputs from all leaf nodes across every built tree:

$$z = z_0 + \sum_{t=1}^{T} \eta \cdot f_t(x)$$

To convert these log-odds into probabilities, the **sigmoid function** is applied:

$$p = \frac{1}{1 + e^{-z}}$$

where:
- $z$ represents the predicted log-odds
- $p$ is the probability that the sample belongs to the positive class

Throughout training, the model updates the accumulated log-odds after every boosting iteration, and the sigmoid function converts the final values into probabilities.

In [12]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1 / (1 + np.exp(-z))

## LogLoss and Taylor Approximation

XGBoost minimizes the **binary cross-entropy (LogLoss)**. Instead of minimizing the exact loss after every new tree, it approximates the loss using a **second-order Taylor expansion** around the current prediction.

The approximation is:

$$
L(y, \hat y + f(x))
\approx
L(y, \hat y)
+
g \cdot f(x)
+
\frac12 h \cdot f(x)^2
$$

where:
- $f(x)$ is the prediction of the newly constructed tree
- $g$ and $h$ denote the **first and second derivatives** of the loss with respect to the current prediction $\hat y$

---

### Why Taylor Approximation is Necessary

In **Logistic Regression**, the model directly optimizes weights $\beta$ in a linear function $z = \beta_0 + \beta_1 x_1 + \dots$, and we can compute gradients with respect to these weights because the structure is fixed and differentiable.

In **XGBoost**, however, $z$ is the cumulative sum of outputs from **all previously built trees**:

$$
z = z_0 + \sum_{t=1}^{T} \eta \cdot f_t(x)
$$

The problem is that **we cannot differentiate the loss with respect to an unbuilt tree structure**. Before a new tree is constructed, we don't know its shape, splits, or leaf values — the structure itself is what we're trying to learn. Therefore, we cannot take the derivative of the loss with respect to the tree's parameters in the same way we do with fixed weights in logistic regression.

Instead, XGBoost uses the **Taylor expansion to approximate the loss** as a simple quadratic function of the new tree's output $f(x)$. This transforms the problem into optimizing a **parabola**:

$$
\text{Approximate loss} \propto a \cdot f(x)^2 + b \cdot f(x) + c
$$

Since this is just a quadratic, the **optimal leaf value** that minimizes the loss can be computed **analytically** using the vertex formula:

$$
f(x)^* = -\frac{b}{2a} = -\frac{g}{h}
$$

This makes optimization extremely efficient — no iterative gradient descent is needed within each boosting round. The model simply:
1. Computes $g$ and $h$ for every sample using the current predictions
2. Builds the tree structure to maximize the gain from splits
3. Sets each leaf's output to $-\frac{g}{h}$ (aggregated over samples in that leaf)

This analytical solution is one of the key reasons XGBoost is so fast and effective.

## Gradients and Hessians

The Taylor approximation requires the first and second derivatives of the loss function with respect to the current log-odds prediction.

The gradient measures the direction in which the prediction should move,

$$
g=p-y,
$$

while the Hessian measures the local curvature of the loss,

$$
h=p(1-p).
$$

For every boosting iteration, these values are computed for every training sample and passed to the decision tree. The tree uses them instead of class labels when searching for the best splits.

In [13]:
def compute_gradients(p: np.ndarray, y: np.ndarray) -> np.ndarray:
    return np.subtract(p, y)

In [14]:
def compute_hessians(p: np.ndarray):
    return np.multiply(p, 1 - p)

## The XGBoost Decision Tree

Unlike traditional decision trees, the XGBoost tree does not evaluate node purity using Gini impurity. Instead, it relies entirely on **gradients** and **Hessians**.

The quality of a node is measured using the **Similarity Score**:

$$
SS = \frac{(\sum g_i)^2}{\sum h_i + \lambda}
$$

Candidate splits are evaluated using the **Gain**:

$$
Gain = SS_{left} + SS_{right} - SS_{parent} - \gamma
$$

A split is **only accepted if `Gain > 0`**, meaning the total similarity score of the two child nodes must exceed the parent's score by at least $\gamma$. In other words, **$Gain$ must be bigger than $\gamma$** — otherwise the split is pruned. This acts as a regularization mechanism: larger $\gamma$ encourages simpler trees by requiring stronger evidence to justify a split.

Finally, every leaf predicts an optimal correction to the current log-odds:

$$
w^* = -\frac{\sum g_i}{\sum h_i + \lambda}
$$

---

### Why This Works: Adaptive Learning Rate via Similarity Score

The **Similarity Score** acts like an **adaptive learning rate** that automatically adjusts based on the local curvature of the loss surface:

- **When the loss surface is flat** (small Hessian, $\sum h_i$ is low), the denominator is small, so $SS$ is large. This means the model takes a **bigger step** — it can be more confident because the gradient direction is reliable.
- **When the loss surface is steep** (large Hessian, $\sum h_i$ is high), the denominator is large, so $SS$ is small. This means the model takes a **smaller step** — it must be more cautious because the gradient may change rapidly.

This adaptive behavior prevents overshooting in regions of high curvature and accelerates progress in flat regions, making optimization more stable and efficient.

---

### The Optimal Leaf Value $w^*$ as the Vertex of the Loss Surface

The leaf value $w^*$ is not just a heuristic — it is the **exact vertex (minimum) of the quadratic Taylor approximation** of the loss surface. Recall that the approximate loss for a leaf is:

$$
\text{Loss} \approx \sum_i \left[ g_i \cdot w + \frac{1}{2} h_i \cdot w^2 \right] + \frac{1}{2} \lambda w^2
$$

This is a **parabola** in $w$ with the form $aw^2 + bw + c$, where:
- $a = \frac{1}{2}(\sum h_i + \lambda)$
- $b = \sum g_i$

The minimum of a parabola is at its **vertex**:

$$
w^* = -\frac{b}{2a} = -\frac{\sum g_i}{\sum h_i + \lambda}
$$

So $w^*$ is literally the **bottom of the loss surface** — the point where the approximate loss is smallest for all samples falling into that leaf. This is why XGBoost can compute optimal leaf values **analytically** without any iterative optimization.






## The Boosting Algorithm

The classifier coordinates the sequential training of multiple XGBoost trees.

Training begins with an initial prediction, typically zero log-odds for every sample. During each boosting iteration, the current log-odds are converted into probabilities, gradients and Hessians are computed, and a new tree is trained to predict the optimal corrections.

The current predictions are then updated according to:

$$ 
z_{\text{new}} = z_{\text{old}} + \eta \cdot f_t(x)
$$

where $\eta$ is the learning rate and $f_t(x)$ denotes the prediction of the current tree.

Each new tree therefore focuses on correcting the mistakes made by the previous ensemble.

---

### From Probabilities to Log-Odds and Back

Log-odds are defined as the **natural logarithm of the odds ratio**:

$$
z = \ln\left(\frac{p}{1 - p}\right)
$$

In one sentence: this formula transforms a probability $p$ (bounded between 0 and 1) into an unbounded real number $z$ by measuring how much more likely the positive class is compared to the negative class on a logarithmic scale — when $p = 0.5$, the odds are 1 and $z = 0$; as $p$ approaches 1, $z$ grows toward $+\infty$; as $p$ approaches 0, $z$ drops toward $-\infty$.

After each boosting step, the updated log-odds are converted back into a new probability using the sigmoid function:

$$
p_{\text{new}} = \sigma(z_{\text{new}}) = \frac{1}{1 + e^{-z_{\text{new}}}} = \frac{1}{1 + e^{-(z_{\text{old}} + \eta \cdot f_t(x))}}
$$

So the new probability is simply the sigmoid of the **sum of all previous log-odds plus the new tree's output** — each tree nudges the probability closer to the true label by adding its correction to the running total of log-odds.

In [15]:
class XGBoostClassifier:
    def __init__(self, n_estimators=10, learning_rate=0.1, max_depth=3,
                 min_samples_split=2, reg_lambda=1.0, gamma=0.0):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.reg_lambda = reg_lambda
        self.gamma = gamma
        self.trees = []
        self.z0 = 0.0  # initial log-odds

    def fit(self, X: pd.DataFrame, y: np.ndarray) -> None:
        # log odds start at 0 - every sample recieves a prediction of 0.5
        log_odds = np.full(shape=y.shape, fill_value=self.z0)

        for _ in range(self.n_estimators):
            p = sigmoid(z=log_odds)
            gradients = compute_gradients(p, y)
            hessians = compute_hessians(p)

            new_tree = XGBoostTree(self.max_depth, self.min_samples_split, self.reg_lambda, self.gamma)
            new_tree.fit(X, gradients, hessians)
            self.trees.append(new_tree)

            new_odds = np.array(new_tree.predict(X))

            log_odds = log_odds + self.learning_rate * new_odds

    def predict_proba(self, X: pd.DataFrame) -> np.ndarray:
        # start from initial log-odds and accumulate corrections from all trees
        log_odds = np.full(shape=X.shape[0], fill_value=self.z0)
        
        for tree in self.trees:
            # predict() returns list of floats, not np.array, so convert it 
            log_odds += self.learning_rate * np.array(tree.predict(X))
        
        # convert final log-odds to probability
        return sigmoid(z=log_odds)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        # convert probabilities to binary predictions (0 or 1)
        probabilities = self.predict_proba(X)
        return (probabilities >= 0.5).astype(int)

## Testing the Classifier

To verify that the implementation works correctly, we will create a small binary classification dataset and train the classifier.

After fitting the model, we will generate predictions and evaluate the classification accuracy. This confirms that the boosting procedure and probability calculations operate as expected before applying the model to larger datasets.

In [16]:
# create the dummy dataset
data = {
    'Age': [22, 25, 47, 35, 14, 50, 28, 19, 60, 38],
    'Sex': ['male', 'female', 'female', 'male', 'male', 'female', 'male', 'female', 'male', 'female'],
    'Survived': [0, 1, 1, 0, 1, 1, 0, 1, 0, 1]
}
df_dummy = pd.DataFrame(data)

X_dummy = df_dummy[['Age', 'Sex']]
y_dummy = df_dummy['Survived'].to_numpy()

# encode categorical feature
X_dummy_encoded = pd.get_dummies(X_dummy, columns=['Sex'], drop_first=True)

# train the XGBoost classifier
clf = XGBoostClassifier(
    n_estimators=10,
    learning_rate=0.3,
    max_depth=2,
    min_samples_split=2,
    reg_lambda=1.0,
    gamma=0.0
)
clf.fit(X_dummy_encoded, y_dummy)

# evaluate the classifier
probabilities = clf.predict_proba(X_dummy_encoded)
predictions = clf.predict(X_dummy_encoded)

print("Probabilities:", np.round(probabilities, 4))
print("Actual:       ", list(y_dummy))
print("Predictions:  ", list(predictions))
print("Accuracy:     ", np.mean(predictions == y_dummy))

Probabilities: [0.1337 0.8829 0.8829 0.1337 0.7193 0.8829 0.1337 0.8829 0.1337 0.8829]
Actual:        [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1)]
Predictions:   [np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1)]
Accuracy:      1.0
